## About
- This notebook includes the eBaseline patient characteristics of the ALD Olink liver inflammation biomarker study.

### Input
1. curated proteomics and clinical data

### Output
1. Baseline patient characteristics tables 1 and 2

In [ ]:
# Load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tableone import TableOne

### Import curated proteomics and clinical data

In [ ]:
# Define project directory
Base = 'data_directory'

df_olink = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='proteomics_curated')
df_cli = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='clinical_curated')

In [ ]:
df_cli_base = df_cli[df_cli['cohort']!='GALA_LiHep'].dropna(subset=['inflam_binary'])

In [ ]:
df_cli_base.groupby('cohort')['supernormal_filter'].value_counts()

In [ ]:
# To Peter
# df_cli_base[df_cli_base['cohort'].isin(['RDC', 'SIP'])][['cohort', 'abstinent']].to_csv('RDC_SIP_SampleIDs.csv')

In [ ]:
discovery_cohorts = ['ALD', 'HP', 'RDC']
df_cli_base[df_cli_base['cohort'].isin(discovery_cohorts)].groupby('inflam_binary')['abstinent'].value_counts(1)

### Baseline patient chracteristics

#### Table 1

In [ ]:
categorical = ['gender', 'aldrisk', 'abstinent', 'hba1c>=48mmol/mol', 'bmi>=30', 'mets', 'steatosis', 'ballooning', 
               'lobinflammation', 'inflammation', 'inflam_binary', 'fibrosis', 'at_risk_progression']
continuous = ['age', 'bmi', ]
nonnormal = ['alt', 'ast', 'ggt', 'hba1c', 'homair', 'insulin', 'trigly', 'hdl', 'ldl', 'te', 'cap', 'fast', 'meld']
groupby = 'cohort'

In [ ]:
columns = categorical + continuous + nonnormal + [groupby]

In [ ]:
rename={'aldrisk':'History of excessive use of alcohol',
       'abstinent':'Abstaining from alcohol at time of inclusion',
        'hba1c>=48mmol/mol':'T2D (HbA1c >= 48mmol/mol)',
        'bmi>=30':'BMI ≥30',
        'mets':'metS',
        'steatosis':'Steatosis 0/1/2/3',
        'ballooning':'Ballooning 0/1/2',
        'lobinflammation':'Lobular inflammation 0/1/2/3',
        'inflammation':'Ballooning + lobular inflammation 0/1/2/3/4/5',
        'inflam_binary':'Mild inflammatory activity (≥I2)',
        'fibrosis':'Fibrosis stage 0-1/2/3/4',
        'at_risk_progression':'Steatohepatitis at risk of progression',
        'age':'Age, y',
        'bmi':'BMI, kg/m2',
        'alt':'ALT, U/L',
        'ast':'AST, U/L',
        'ggt':'GGT, U/L',
        'hba1c':'HbA1c, mmol/mol',
        'homair':'HOMA-IR',
        'insulin':'Insulin, pmol/L',
        'trigly':'Triglycerides, mmol/L',
        'hdl':'HDL cholesterol, mmol/L',
        'ldl':'LDL cholesterol, mmol/L',
        'te':'TE, kPa',
        'cap':'CAP',
        'fast':'FAST',
        'meld':'MELD'
       }

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_table=df_cli_base.copy()
table1 = TableOne(df_table, columns=columns, categorical=categorical, continuous = [],
                  groupby=groupby, nonnormal=nonnormal, rename=rename, pval=False)

In [ ]:
pd.set_option('display.max_rows', 500)

#### Table 2

In [ ]:
categorical = ['gender']
continuous = ['age', 'bmi', ]
nonnormal = ['alt', 'ast', 'ggt', 'hba1c', 'homair', 'insulin', 'trigly', 'hdl', 'ldl', 'te']
groupby = 'inflam_binary'

In [ ]:
df_cli_base['cohort_group1'] = df_cli_base['cohort'].map({'ALD':'derivation', 
                                                          'HP':'derivation',
                                                          'RDC':'derivation',
                                                          'SIP':'validation'
                                                         })

In [ ]:
columns = categorical + continuous + nonnormal + [groupby]

In [ ]:
df_table2_deri = df_cli_base[df_cli_base['cohort_group1']=='derivation']
df_table2_vali = df_cli_base[df_cli_base['cohort_group1']=='validation']

table2_deri = TableOne(df_table2_deri, columns=columns, categorical=categorical, continuous = [],
                  groupby=groupby, nonnormal=nonnormal, rename=rename, pval=False)
table2_vali = TableOne(df_table2_vali, columns=columns, categorical=categorical, continuous = [],
                  groupby=groupby, nonnormal=nonnormal, rename=rename, pval=False)

### Save summary demograpics 

In [ ]:
# Save baseline patient characteristics in table 1 and 2
with pd.ExcelWriter(os.path.join(Base, 'tables/baseline_patient_characteristics.xlsx')) as writer:
    table1.to_excel(writer, sheet_name='table1')
    table2_deri.to_excel(writer, sheet_name='table2_derivation') 
    table2_vali.to_excel(writer, sheet_name='table2_validation') 

### Explore normal and supernormal classifers

In [ ]:
df_cli_base.groupby(['biopsy', 'cohort', 'normal'])['inflam_binary'].value_counts(0).round(2)

- Problem: 
> 34% (281/829) had no biopsy and were assigned to ‘healthy’ based on a set of criteria. However, based on this definition, using biopsy-verified disease stage as ground truth, up to 30% of the participants classified as ‘healthy’ (29% and 12% in the ALD and RDC cohort, respectively) have mild inflammation. 
- Consequence:
> Imperfectly labeled training data compromises the model’s discriminative power due to ‘weaker class separation’. 
- Solution: 
> Question 1: How the already-trained model, built under noisier labels, generalizes when evaluated against a cleaner/more accurate definition of ”healthy”.
> 
> Approach: apply a more stringent criteria to only the testing and validation sets.
> 
> Question 2: How the model would perform if the ground truth definition of ”healthy" had been stricter from the start by eliminating label noise in training
> 
> Approach: apply a more stringent criteria to all sets (training, testing and validation)


In [ ]:
df_cli_base[~df_cli_base['supernormal_filter']].groupby(['cohort_group1'])['inflam_binary'].value_counts(1)

In [ ]:
df_cli_base.groupby(['cohort_group1'])['inflam_binary'].value_counts(1)

In [ ]:
df_cli_base[~df_cli_base['supernormal_filter']].groupby('cohort_group1')['inflam_binary'].value_counts(1)

In [ ]:
df_cli_base.groupby('cohort')[['alt','ast','crp', 'inflammation', 'kleiner', 'aldrisk', 'inflam_binary', 'fast']].count()

In [ ]:
df_table['group']=np.where(df_table['cohort'].isin(['ALD', 'HP', 'RDC']), 'derivation', 'validation')

In [ ]:
colors = ['gray', 'steelblue']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Boxplot
sns.boxplot(x='group', y='ast_log2', hue='inflam_binary', data=df_table, ax=axes[0], palette=colors)
axes[0].legend(title='mild inflammatory\n     activity (≥I2)', bbox_to_anchor=(1.05, 1), loc='upper left')

# Bar plot
df_toplot = (df_table[df_table['inflammation'] != 'None']
             .groupby('group')['inflammation']
             .value_counts(normalize=True)
             .rename('proportion')
             .reset_index()
             .pivot(columns='group', index='inflammation')['proportion'])
df_toplot.plot(kind='bar', rot=0, ax=axes[1], color=colors)
axes[1].legend(title='cohort', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].set_xlabel('')
axes[0].set_ylabel('AST [log2]')
axes[1].set_ylabel('Proportion within group\n(histologically staged only)')
axes[1].set_xlabel('ballooning + lobular inflammation')

plt.tight_layout()
plt.savefig('figures/figure_Sx.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
import pingouin as pg

In [ ]:
df_table[df_table['inflam_binary']==1].groupby('group')['ast'].median()

In [ ]:
pg.ttest(x=df_table[(df_table['group']=='derivation') & (df_table['inflam_binary']==1)]['ast_log2'], 
         y=df_table[(df_table['group']=='validation') & (df_table['inflam_binary']==1)]['ast_log2'])

In [ ]:
alcohol = pd.read_csv(os.path.join(Base, 'raw/abstinence/INF2_validationID.csv'))

In [ ]:
df_table = df_table.set_index('CBMRID').join(alcohol.set_index('id'), how='left')
df_table.groupby(['group', 'inflam_binary'])['alc_group'].value_counts(1)

In [ ]:
df_table.groupby(['group', 'inflam_binary'])['at_risk_progression'].value_counts()